# Machine Learning Training and SHAP Analysis for ZIF-8 Synthesis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import LeaveOneOut, GridSearchCV, cross_val_predict, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

import shap
import time
import json

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 100,
    'savefig.dpi': 100,
    'pdf.fonttype': 42,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Liberation Sans', 'Bitstream Vera Sans'],
    'font.weight': 'bold',
    'font.style' : 'normal'
    #'svg.fonttype' : 'none'

})
plt.rcParams['pdf.fonttype'] = 42  
plt.rcParams['svg.fonttype'] = 'none'

model_output_dir = Path('ml_models')
model_output_dir.mkdir(exist_ok=True)
shap_viz_dir = Path('shap_visualizations')
shap_viz_dir.mkdir(exist_ok=True)

## Load and Prepare Data

In [ ]:
df = pd.read_csv('fit_data.csv')
df_clean = df.dropna(subset=['size_nm'])
            
            
    

print(f"Dataset shape: {df_clean.shape}")
print(f"\nTarget variable (size_nm) statistics:\n{df_clean['size_nm'].describe()}")

num_features = ['temperature_C', 'time_min', 'hmim_mM', 'zn_source_mM', 'ratio']
cat_features = ['counterion', 'solvent_name']
ord_features = ['stirring']

print(f"\nNumerical features: {num_features}")
print(f"Categorical features: {cat_features}")
print(f"Ordinal features: {ord_features}")
print(f"\nSolvent values: {df_clean['solvent_name'].value_counts()}")
print(f"\nStirring values: {df_clean['stirring'].value_counts()}")

## Feature Engineering and Model Setup

In [ ]:
from sklearn.compose import ColumnTransformer
X = df_clean[num_features + cat_features + ord_features]
y = df_clean['size_nm']

stirring_order = [ 'No', 'Initial', 'Yes', 'Yes dropwise']
stirring_order = [s for s in stirring_order if s in df_clean['stirring'].unique()]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_features),
        ('ord', OrdinalEncoder(categories=[stirring_order], handle_unknown='use_encoded_value', unknown_value=-1), ord_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ],
    remainder='passthrough')

X_preprocessed = preprocessor.fit_transform(X)
feature_names = (num_features + ord_features + 
                list(preprocessor.named_transformers_['cat'].get_feature_names_out(cat_features)))

print(f"Preprocessed X shape: {X_preprocessed.shape}")
print(f"Number of features after preprocessing: {len(feature_names)}")
print(f"Target variable shape: {y.shape}")

# Save preprocessor for future inference
joblib.dump(preprocessor, model_output_dir / 'preprocessor.pkl')
print(f"\n✓ Preprocessor saved to: {model_output_dir / 'preprocessor.pkl'}")

## Model Training with Extended Parameter Grid

In [ ]:
rf_model = RandomForestRegressor(random_state=42)


param_grid = {
    'n_estimators': [300],
    'max_depth': [15],
    'min_samples_split': [2]
}

print(f"Parameter grid combinations: {np.prod([len(v) for v in param_grid.values()])}")

loo = LeaveOneOut()

start_time = time.time()
grid_search = GridSearchCV(
    rf_model,
    param_grid,
    cv=loo,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

print("Starting GridSearchCV with LeaveOneOut validation...")
grid_search.fit(X_preprocessed, y)
elapsed_time = time.time() - start_time

print(f"\nTraining completed in {elapsed_time:.2f} seconds")
print(f"Best parameters: {grid_search.best_params_}")
best_rmse = np.sqrt(-grid_search.best_score_)
print(f"Best RMSE (LOOCV): {best_rmse:.4f}")

best_model = grid_search.best_estimator_
joblib.dump(best_model, model_output_dir / 'final_optimized_model.pkl')
joblib.dump(grid_search, model_output_dir / 'grid_search_results.pkl')
print(f"\nModels saved to {model_output_dir.absolute()}")

## Model Evaluation Metrics

In [ ]:
best_model = joblib.load('ml_models/final_optimized_model.pkl')
loo = LeaveOneOut()
y_pred_loocv = cross_val_predict(best_model, X_preprocessed, y, cv=loo)

rmse_loocv = np.sqrt(mean_squared_error(y, y_pred_loocv))
mae_loocv = mean_absolute_error(y, y_pred_loocv)
r2_loocv = r2_score(y, y_pred_loocv)

y_pred_train = best_model.predict(X_preprocessed)
rmse_train = np.sqrt(mean_squared_error(y, y_pred_train))
r2_train = r2_score(y, y_pred_train)

print("=== Model Performance ===")
print(f"LOOCV RMSE: {rmse_loocv:.4f}")
print(f"LOOCV MAE: {mae_loocv:.4f}")
print(f"LOOCV R²: {r2_loocv:.4f}")
print(f"Training RMSE: {rmse_train:.4f}")
print(f"Training R²: {r2_train:.4f}")

print(f"\n=== Optimized Model Parameters ===")
# for param, value in grid_search.best_params_.items():
#     print(f"{param}: {value}")

residuals = y - y_pred_loocv
print(f"\n=== Residual Statistics ===")
print(f"Mean: {residuals.mean():.4f}")
print(f"Std: {residuals.std():.4f}")
print(f"Min: {residuals.min():.4f}")
print(f"Max: {residuals.max():.4f}")

## SHAP Analysis and Feature Importance

In [ ]:
print("Computing SHAP values (TreeExplainer)...")
best_model = joblib.load('ml_models/final_optimized_model.pkl')
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_preprocessed)

print(f"SHAP values shape: {shap_values.shape}")
expected_value = explainer.expected_value if isinstance(explainer.expected_value, (int, float)) else explainer.expected_value[0]
print(f"Base value: {expected_value:.4f}")

feature_importance_mean = np.abs(shap_values).mean(axis=0)
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Mean |SHAP|': feature_importance_mean
}).sort_values('Mean |SHAP|', ascending=False)

print("\n=== SHAP Feature Importance ===")
print(feature_importance_df.head(15))

joblib.dump(explainer, model_output_dir / 'shap_explainer.pkl')
np.save(model_output_dir / 'shap_values.npy', shap_values)

## Academic Visualization Framework with Integrated Data Export

The `MLVisualizer` class provides a unified framework for:
- **Figure Generation**: Creates publication-quality visualizations in SVG (editable), PDF (vector), and PNG (raster) formats
- **Automatic Data Export**: Simultaneously saves raw data in JSON and CSV formats
- **Standardized Styling**: Consistent academic formatting across all plots
- **Feature Group Analysis**: Integrated support for aggregated feature importance visualization

### Key Features
1. **Multi-format Export**: Each visualization automatically saves in 3 formats
2. **Built-in Data Logging**: Raw data is captured and exported alongside figures
3. **Scalable Design**: New visualization methods can be added easily
4. **Comprehensive SHAP Analysis**: Full integration with SHAP framework including dependence plots, decision plots, and feature group analysis

In [ ]:
class MLVisualizer:
    """Academic visualization framework with integrated data export"""
    
    def __init__(self, output_dir=Path('shap_visualizations'), data_export_dir=Path('visualization_data')):
        self.output_dir = output_dir
        self.data_export_dir = data_export_dir
        self.output_dir.mkdir(exist_ok=True)
        self.data_export_dir.mkdir(exist_ok=True)
        self.fig_count = 0
        self.exported_data = {}
    
    def _setup_figure(self, figsize=(8, 5)):
        fig, ax = plt.subplots(figsize=figsize)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        return fig, ax
    
    def _save_figure(self, fig, name, data=None, data_format='json'):
        """Save figure in multiple formats (SVG, PDF, PNG) + optional raw data
        
        Args:
            fig: matplotlib figure object
            name: base name for saved files
            data: dict with data to export (optional)
            data_format: 'json', 'csv', or 'both' (default: 'json')
        """
        # Save figure in multiple formats
        formats = {
            'svg': {'format': 'svg', 'dpi': 300},
            'pdf': {'format': 'pdf', 'dpi': 300},
            'png': {'format': 'png', 'dpi': 300}
        }
        
        for ext, params in formats.items():
            path = self.output_dir / f"{name}.{ext}"
            fig.savefig(path, transparent=True, bbox_inches='tight', **params)
        
        # Save raw data if provided
        if data is not None:
            self._export_data(name, data, data_format)
        
        plt.close(fig)
        self.fig_count += 1
    
    def _export_data(self, name, data, format_type='json'):
        """Export data in specified format(s)"""
        if format_type in ['json', 'both']:
            json_path = self.data_export_dir / f"{name}.json"
            with open(json_path, 'w') as f:
                json.dump(data, f, indent=2)
        
        if format_type in ['csv', 'both'] and isinstance(data, dict) and len(data) > 0:
            # Try to convert dict to DataFrame for CSV export
            try:
                if all(isinstance(v, list) and len(v) > 0 for v in data.values()):
                    df = pd.DataFrame(data)
                    csv_path = self.data_export_dir / f"{name}.csv"
                    df.to_csv(csv_path, index=False)
            except:
                pass  # If can't convert, skip CSV
    
    def predictions_vs_actual(self, y_true, y_pred):
        """Predictions vs actual scatter plot with R² metric"""
        r2 = r2_score(y_true, y_pred)
        
        fig, ax = self._setup_figure()
        ax.scatter(y_true, y_pred, alpha=0.6, s=50, color='steelblue', edgecolors='navy', linewidth=0.3)
        min_val, max_val = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
        ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5, alpha=0.7)
        ax.set_xlabel('Actual Size (nm)')
        ax.set_ylabel('Predicted Size (nm)')
        ax.text(0.05, 0.95, f'R² = {r2:.4f}', transform=ax.transAxes,
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        
        # Export data
        data = {
            'y_actual': y_true.tolist() if hasattr(y_true, 'tolist') else list(y_true),
            'y_predicted': y_pred.tolist() if hasattr(y_pred, 'tolist') else list(y_pred),
            'r2_score': float(r2),
        }
        self._save_figure(fig, '01_predictions_vs_actual', data, 'both')
    
    def residuals_distribution(self, y_true, y_pred):
        """Residuals distribution histogram"""
        residuals = y_true - y_pred
        
        fig, ax = self._setup_figure()
        ax.hist(residuals, bins=25, color='steelblue', alpha=0.7, edgecolor='black', linewidth=0.5)
        ax.axvline(0, color='red', linestyle='--', linewidth=1.5)
        ax.set_xlabel('Residuals (nm)')
        ax.set_ylabel('Frequency')
        
        # Export data
        data = {
            'residuals': residuals.tolist() if hasattr(residuals, 'tolist') else list(residuals),
            'actual': y_true.tolist() if hasattr(y_true, 'tolist') else list(y_true),
            'predicted': y_pred.tolist() if hasattr(y_pred, 'tolist') else list(y_pred),
        }
        self._save_figure(fig, '02_residuals_distribution', data, 'both')
    
    def residuals_vs_predicted(self, y_true, y_pred):
        """Residuals vs predicted values scatter plot"""
        residuals = y_true - y_pred
        
        fig, ax = self._setup_figure()
        ax.scatter(y_pred, residuals, alpha=0.6, s=50, color='steelblue', edgecolors='navy', linewidth=0.3)
        ax.axhline(0, color='red', linestyle='--', linewidth=1.5)
        ax.set_xlabel('Predicted Size (nm)')
        ax.set_ylabel('Residuals (nm)')
        
        # Export data
        data = {
            'predicted': y_pred.tolist() if hasattr(y_pred, 'tolist') else list(y_pred),
            'residuals': residuals.tolist() if hasattr(residuals, 'tolist') else list(residuals),
        }
        self._save_figure(fig, '03_residuals_vs_predicted', data, 'both')
    
    def feature_importance(self, importance_df, top_n=15):
        """SHAP feature importance bar plot"""
        fig, ax = self._setup_figure()
        top_features = importance_df.head(top_n)
        ax.barh(range(len(top_features)), top_features['Mean |SHAP|'], color='steelblue', edgecolor='black', linewidth=0.5)
        ax.set_yticks(range(len(top_features)))
        ax.set_yticklabels(top_features['Feature'], fontsize=9)
        ax.set_xlabel('Mean |SHAP value|')
        ax.invert_yaxis()
        
        # Export data
        data = top_features[['Feature', 'Mean |SHAP|']].to_dict('records')
        self._save_figure(fig, '04_feature_importance_shap', data, 'json')
    
    def error_distribution(self, y_true, y_pred):
        """Absolute error distribution histogram"""
        errors = np.abs(y_true - y_pred)
        
        fig, ax = self._setup_figure()
        ax.hist(errors, bins=25, color='lightcoral', alpha=0.7, edgecolor='black', linewidth=0.5)
        ax.set_xlabel('Absolute Error (nm)')
        ax.set_ylabel('Frequency')
        ax.axvline(errors.mean(), color='darkred', linestyle='--', linewidth=1.5, label=f'Mean: {errors.mean():.2f}')
        ax.legend(frameon=True, fancybox=False, edgecolor='black')
        
        # Export data
        data = {
            'absolute_errors': errors.tolist() if hasattr(errors, 'tolist') else list(errors),
            'mean_error': float(np.mean(errors)),
            'std_error': float(np.std(errors)),
        }
        self._save_figure(fig, '05_absolute_error_distribution', data, 'both')
    
    def model_cv_scores(self, cv_results):
        """Cross-validation scores across configurations"""
        fig, ax = self._setup_figure()
        mean_scores = -cv_results['mean_test_score']
        std_scores = cv_results['std_test_score']
        
        ax.bar(range(len(mean_scores[:20])), mean_scores[:20], yerr=std_scores[:20],
               color='steelblue', alpha=0.7, edgecolor='black', linewidth=0.5, capsize=3)
        ax.set_xlabel('Hyperparameter Configuration')
        ax.set_ylabel('MSE')
        ax.set_xticks(range(0, len(mean_scores[:20]), 2))
        
        # Export data
        data = {
            'mean_scores': mean_scores.tolist(),
            'std_scores': std_scores.tolist(),
            'params': [str(p) for p in cv_results['params']],
        }
        self._save_figure(fig, '06_cv_scores_progression', data, 'json')
    
    def predictions_vs_feature(self, feature_values, y_true, y_pred, feature_name, filename_prefix, use_log_scale=False):
        """Predictions vs physical parameter (e.g., temperature, time)"""
        fig, ax = self._setup_figure()
        ax.scatter(feature_values, y_pred, alpha=0.6, s=50, color='steelblue', 
                  edgecolors='navy', linewidth=0.3, label='Predicted')
        ax.scatter(feature_values, y_true, alpha=0.4, s=30, color='red', marker='x', 
                  linewidth=1.5, label='Actual')
        
        ax.set_xlabel(feature_name)
        ax.set_ylabel('Size (nm)')
        if use_log_scale:
            ax.set_xscale('log')
        ax.legend(frameon=True, fancybox=False, edgecolor='black')
        
        # Export data
        data = {
            feature_name.replace(' ', '_').lower(): feature_values.tolist() if hasattr(feature_values, 'tolist') else list(feature_values),
            'y_actual': y_true.tolist() if hasattr(y_true, 'tolist') else list(y_true),
            'y_predicted': y_pred.tolist() if hasattr(y_pred, 'tolist') else list(y_pred),
        }
        self._save_figure(fig, filename_prefix, data, 'both')
    
    def feature_group_importance(self, group_importance_df, shap_values, feature_names, group_indices):
        """Feature group importance visualization with breakdown data"""
        fig, ax = self._setup_figure(figsize=(10, 6))
        colors = plt.cm.Blues(np.linspace(0.3, 0.8, len(group_importance_df)))
        ax.barh(range(len(group_importance_df)), group_importance_df['Mean |SHAP|'], 
               color=colors, edgecolor='black', linewidth=0.7)
        
        ax.set_yticks(range(len(group_importance_df)))
        ax.set_yticklabels(group_importance_df['Feature Group'], fontsize=11, fontweight='bold')
        ax.set_xlabel('Mean |SHAP value|', fontsize=12, fontweight='bold')
        ax.set_title('Feature Group Importance for ZIF-8 Particle Size Prediction', 
                     fontsize=13, fontweight='bold', pad=15)
        ax.invert_yaxis()
        
        # Add value labels on bars
        for i, (idx, row) in enumerate(group_importance_df.iterrows()):
            ax.text(row['Mean |SHAP|'] + 0.3, i, f"{row['Mean |SHAP|']:.4f}", 
                    va='center', fontsize=10, fontweight='bold')
        
        # Prepare data for export
        data = group_importance_df[['Feature Group', 'Mean |SHAP|']].to_dict('records')
        
        # Create detailed breakdown
        group_breakdown = {}
        for group_name, indices in sorted(group_indices.items()):
            if indices:
                constituent_features = [feature_names[i] for i in indices]
                group_shap_val = group_importance_df[group_importance_df['Feature Group'] == group_name]['Mean |SHAP|'].values[0]
                feature_details = {}
                for feat in constituent_features:
                    feat_idx = feature_names.index(feat)
                    feat_shap = np.abs(shap_values[:, feat_idx]).mean()
                    feature_details[feat] = float(feat_shap)
                group_breakdown[group_name] = {
                    'total_importance': float(group_shap_val),
                    'num_features': len(constituent_features),
                    'feature_details': feature_details
                }
        
        # Save with breakdown data
        self._save_figure(fig, '07_feature_group_importance', data, 'json')
        
        # Also save breakdown separately
        breakdown_path = self.data_export_dir / '07_feature_group_breakdown.json'
        with open(breakdown_path, 'w') as f:
            json.dump(group_breakdown, f, indent=2)
        
        print(f"\n✓ Feature group importance visualization and breakdown saved")
        return group_breakdown

visualizer = MLVisualizer(shap_viz_dir, Path('visualization_data'))

## Generate Model Performance Visualizations

In [ ]:
import json
visualizer.predictions_vs_actual(y.values, y_pred_loocv)
visualizer.residuals_distribution(y.values, y_pred_loocv)
visualizer.residuals_vs_predicted(y.values, y_pred_loocv)
visualizer.error_distribution(y.values, y_pred_loocv)
visualizer.feature_importance(feature_importance_df)
visualizer.model_cv_scores(grid_search.cv_results_)

print(f"Model performance visualizations generated: {visualizer.fig_count}")

## SHAP Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plt.sca(ax)
shap.summary_plot(shap_values, X_preprocessed, feature_names=feature_names, plot_type="bar", show=False)
ax.figure.savefig(shap_viz_dir / '07_shap_summary_bar.svg', format='svg', dpi=300, transparent=True, bbox_inches='tight')
ax.figure.savefig(shap_viz_dir / '07_shap_summary_bar.pdf', format='pdf', dpi=300, transparent=True, bbox_inches='tight')
ax.figure.savefig(shap_viz_dir / '07_shap_summary_bar.png', format='png', dpi=300, transparent=True, bbox_inches='tight')
plt.close()

fig, ax = plt.subplots(figsize=(10, 8))
plt.sca(ax)
shap.summary_plot(shap_values, X_preprocessed, feature_names=feature_names, show=False)
ax.figure.savefig(shap_viz_dir / '08_shap_summary_bee.svg', format='svg', dpi=300, transparent=True, bbox_inches='tight')
ax.figure.savefig(shap_viz_dir / '08_shap_summary_bee.pdf', format='pdf', dpi=300, transparent=True, bbox_inches='tight')
ax.figure.savefig(shap_viz_dir / '08_shap_summary_bee.png', format='png', dpi=300, transparent=True, bbox_inches='tight')
plt.close()

print("SHAP summary plots generated")

In [ ]:
top_features_idx = feature_importance_df.head(6).index.values
for idx, feature_idx in enumerate(top_features_idx):
    feature_name = feature_names[feature_idx]
    # SHAP dependence_plot creates its own figure, so we capture it with gcf()
    plt.figure(figsize=(8, 5))
    shap.dependence_plot(feature_idx, shap_values, X_preprocessed, feature_names=feature_names, show=False)
    fig = plt.gcf()  # Get the figure that SHAP just created
    fig.savefig(shap_viz_dir / f'09_shap_dependence_{idx:02d}_{feature_name}.svg', format='svg', dpi=300, transparent=True, bbox_inches='tight')
    fig.savefig(shap_viz_dir / f'09_shap_dependence_{idx:02d}_{feature_name}.pdf', format='pdf', dpi=300, transparent=True, bbox_inches='tight')
    fig.savefig(shap_viz_dir / f'09_shap_dependence_{idx:02d}_{feature_name}.png', format='png', dpi=300, transparent=True, bbox_inches='tight')
    plt.close()

print(f"SHAP dependence plots for top {len(top_features_idx)} features generated with proper rendering")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
shap.decision_plot(expected_value, shap_values, X_preprocessed, feature_names=feature_names, show=False)

# ax.figure.savefig(shap_viz_dir / '10_shap_decision_plot.svg', format='svg', dpi=300, transparent=True, bbox_inches='tight')
# ax.figure.savefig(shap_viz_dir / '10_shap_decision_plot.pdf', format='pdf', dpi=300, transparent=True, bbox_inches='tight')
# ax.figure.savefig(shap_viz_dir / '10_shap_decision_plot.png', format='png', dpi=300, transparent=True, bbox_inches='tight')
# plt.close()

print("SHAP decision plot generated")

## Additional Feature and Prediction Visualizations

In [ ]:
# Additional feature-based prediction visualizations with integrated data export
print("Generating feature-based prediction visualizations...\n")

# Temperature vs Predictions
visualizer.predictions_vs_feature(
    df_clean['temperature_C'].values,
    y.values,
    y_pred_loocv,
    'Temperature (°C)',
    '08_predictions_vs_temperature',
    use_log_scale=False
)
print("✓ Temperature vs predictions")

# Time vs Predictions (log scale)
visualizer.predictions_vs_feature(
    df_clean['time_min'].values,
    y.values,
    y_pred_loocv,
    'Time (min)',
    '09_predictions_vs_time',
    use_log_scale=True
)
print("✓ Time vs predictions (log scale)")

print(f"\nFeature-based visualizations completed")

## Complete Model Summary and Export Verification

In [ ]:
# Aggregate SHAP values by feature groups
print("="*70)
print("FEATURE GROUP AGGREGATION AND VISUALIZATION")
print("="*70)

# Create a mapping of feature indices to their groups
feature_groups = {}
group_indices = {}

# Numerical features
for feat in num_features:
    idx = feature_names.index(feat)
    feature_groups[idx] = feat
    group_indices[feat] = [idx]

# Ordinal features
for feat in ord_features:
    idx = feature_names.index(feat)
    feature_groups[idx] = feat
    group_indices[feat] = [idx]

# Categorical features (one-hot encoded)
for cat_feat in cat_features:
    group_indices[cat_feat] = []
    for feat_name in feature_names:
        if feat_name.startswith(cat_feat + '_'):
            idx = feature_names.index(feat_name)
            feature_groups[idx] = cat_feat
            group_indices[cat_feat].append(idx)

# Calculate mean |SHAP| for each group
group_importance = {}
for group_name, indices in group_indices.items():
    if indices:  # Only if group has features
        group_shap = np.abs(shap_values[:, indices]).mean()
        group_importance[group_name] = group_shap

# Convert to DataFrame and sort
group_importance_df = pd.DataFrame(list(group_importance.items()), 
                                   columns=['Feature Group', 'Mean |SHAP|'])
group_importance_df = group_importance_df.sort_values('Mean |SHAP|', ascending=False)

print("\n=== SHAP Feature Group Importance ===\n")
print(group_importance_df.to_string(index=False))

# Generate visualization using MLVisualizer
group_breakdown = visualizer.feature_group_importance(
    group_importance_df, 
    shap_values, 
    feature_names, 
    group_indices
)

# Print detailed breakdown
print("\n=== Feature Group Breakdown ===")
for group_name, details in sorted(group_breakdown.items()):
    print(f"\n{group_name}:")
    print(f"  Importance (Mean |SHAP|): {details['total_importance']:.4f}")
    print(f"  Number of features: {details['num_features']}")
    if details['num_features'] <= 3:
        for feat, importance in details['feature_details'].items():
            print(f"    • {feat}: {importance:.4f}")
    else:
        features_list = list(details['feature_details'].items())
        for feat, importance in features_list[:3]:
            print(f"    • {feat}: {importance:.4f}")
        print(f"    + {details['num_features'] - 3} more...")

## Feature Group Importance Analysis

Aggregating SHAP values by feature groups (numerical, ordinal, and categorical) to provide high-level feature importance insights.

In [ ]:
import glob

print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"\nDataset: fit_data.csv")
print(f"Samples: {len(df_clean)}")
print(f"Target: Particle Size (nm)")
print(f"Features: {len(feature_names)}")

print(f"\n=== Model Configuration ===")
print(f"Base Model: RandomForestRegressor")
print(f"Best Parameters: {grid_search.best_params_}")

print(f"\n=== Model Performance ===")
print(f"LOOCV RMSE: {rmse_loocv:.4f} nm")
print(f"LOOCV MAE: {mae_loocv:.4f} nm")
print(f"LOOCV R²: {r2_loocv:.4f}")
print(f"Training RMSE: {rmse_train:.4f} nm")
print(f"Training R²: {r2_train:.4f}")

print(f"\n=== Saved Model Files ===")
for f in model_output_dir.glob('*'):
    size = f.stat().st_size / 1024
    print(f"{f.name}: {size:.2f} KB")

print(f"\n=== Generated Visualizations ===")
formats = ['svg', 'pdf', 'png']
total_files = 0
for fmt in formats:
    files = glob.glob(f"{shap_viz_dir}/*.{fmt}")
    count = len(files)
    total_files += count
    print(f"{fmt.upper()}: {count} files")

print(f"\nTotal visualization files: {total_files}")
print(f"Estimated disk space: {total_files * 100 / 1024:.2f} MB")
print(f"Output directory: {shap_viz_dir.absolute()}")
print(f"Model directory: {model_output_dir.absolute()}")

In [ ]:
import json
import glob

data_export_dir = Path('visualization_data')

print("="*70)
print("SUPPLEMENTARY DATA EXPORT")
print("="*70)

# 1. SHAP Feature Importance (individual features)
print("\n[1/5] Exporting individual SHAP feature importance...")
feature_importance_df.to_csv(data_export_dir / '02_shap_feature_importance.csv', index=False)
with open(data_export_dir / '02_shap_feature_importance.json', 'w') as f:
    json.dump(feature_importance_df.to_dict('records'), f, indent=2)
print("  ✓ 02_shap_feature_importance.json")
print("  ✓ 02_shap_feature_importance.csv")

# 2. SHAP Dependence Plot Data
print("\n[2/5] Exporting SHAP dependence plot data...")
dependence_data = {}
top_features_idx_list = feature_importance_df.head(6).index.values
for idx, feature_idx in enumerate(top_features_idx_list):
    feature_name = feature_names[feature_idx]
    dependence_data[feature_name] = {
        'feature_values': X_preprocessed[:, feature_idx].tolist(),
        'shap_values': shap_values[:, feature_idx].tolist(),
    }

with open(data_export_dir / '08_shap_dependence_data.json', 'w') as f:
    json.dump(dependence_data, f, indent=2)
print("  ✓ 08_shap_dependence_data.json")

# 3. SHAP Decision Plot Data
print("\n[3/5] Exporting SHAP decision plot data...")
decision_data = {
    'base_value': float(expected_value),
    'shap_values_samples': shap_values[:20].tolist(),
    'feature_names': feature_names,
    'sample_predictions': y_pred_loocv[:20].tolist(),
}
with open(data_export_dir / '09_shap_decision_data.json', 'w') as f:
    json.dump(decision_data, f, indent=2)
print("  ✓ 09_shap_decision_data.json")

# 4. SHAP Summary Statistics
print("\n[4/5] Exporting SHAP summary statistics...")
summary_data = {
    'feature_names': feature_names,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0).tolist(),
    'base_value': float(expected_value),
}
with open(data_export_dir / '10_shap_summary_data.json', 'w') as f:
    json.dump(summary_data, f, indent=2)
print("  ✓ 10_shap_summary_data.json")

# 5. Model Configuration and Metadata
print("\n[5/5] Exporting model configuration...")
model_config = {
    'best_parameters': {k: int(v) if isinstance(v, np.integer) else v for k, v in grid_search.best_params_.items()},
    'performance_metrics': {
        'loocv_rmse': float(rmse_loocv),
        'loocv_mae': float(mae_loocv),
        'loocv_r2': float(r2_loocv),
        'training_rmse': float(rmse_train),
        'training_r2': float(r2_train),
    },
    'dataset_info': {
        'total_samples': len(df_clean),
        'num_features': len(feature_names),
        'feature_names': feature_names,
    },
    'cv_strategy': 'LeaveOneOut',
    'base_model': 'RandomForestRegressor',
}
with open(data_export_dir / '99_model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)
print("  ✓ 99_model_config.json")

print("\n" + "="*70)
print("DATA EXPORT SUMMARY")
print("="*70)

# List all files with sizes
data_files = sorted(Path('visualization_data').glob('*'))
print(f"\nTotal files: {len(data_files)}")

formats_count = {'json': 0, 'csv': 0, 'other': 0}
total_size = 0
for f in data_files:
    size = f.stat().st_size / 1024
    total_size += size
    ext = f.suffix.lstrip('.')
    if ext in formats_count:
        formats_count[ext] += 1
    else:
        formats_count['other'] += 1

print(f"JSON files: {formats_count['json']}")
print(f"CSV files: {formats_count['csv']}")
print(f"Total disk space: {total_size / 1024:.2f} MB")
print(f"\nExport location: {data_export_dir.absolute()}")

## Data Files Documentation

All visualization data is **automatically exported** by the `MLVisualizer` class in both JSON and CSV formats:
- **JSON Format**: For programmatic processing and structured data analysis
- **CSV Format**: For Excel, Pandas, and statistical software

### Generated Data Files

| Visualization ID | Data Files | Content | Format |
|-----------------|-----------|---------|--------|
| 01 Predictions vs Actual | `.json, .csv` | Predicted vs actual values, R² score | Both |
| 02 Residuals Distribution | `.json, .csv` | Residuals, actual, predicted values | Both |
| 03 Residuals vs Predicted | `.json, .csv` | Predicted values and residuals | Both |
| 04 Feature Importance SHAP | `.json` | Individual feature SHAP importance | JSON |
| 05 Absolute Error Distribution | `.json, .csv` | Error values and statistics | Both |
| 06 CV Scores | `.json` | Cross-validation scores across configs | JSON |
| 07 Feature Group Importance | `.json` | Aggregated feature group importance | JSON |
| 07 Feature Group Breakdown | `.json` | Detailed feature composition by group | JSON |
| 08 Temperature Predictions | `.json, .csv` | Temperature vs size predictions | Both |
| 09 Time Predictions | `.json, .csv` | Reaction time vs size predictions | Both |
| 02 Individual Features | `.csv` | All individual feature importance | CSV |
| 08 SHAP Dependence Data | `.json` | Feature values vs SHAP values | JSON |
| 09 SHAP Decision Data | `.json` | Sample-level decision path data | JSON |
| 10 SHAP Summary Stats | `.json` | Feature names and summary statistics | JSON |
| 99 Model Config | `.json` | Model parameters and performance metrics | JSON |

In [ ]:
print("\n" + "="*80)
print("DATA FILES SUMMARY - All visualization data exported successfully")
print("="*80)

# Show data directory structure
data_files = sorted(Path('visualization_data').glob('*'))
print(f"\nData directory location: {Path('visualization_data').absolute()}")
print(f"Total files: {len(data_files)}")
print(f"Total size: {sum(f.stat().st_size for f in data_files) / 1024:.1f} KB\n")

print("File List:")
print("-" * 80)
for f in data_files:
    size = f.stat().st_size / 1024
    file_type = f.suffix.upper() if f.suffix else 'NO-EXT'
    print(f"  {f.name:40s} | {size:8.2f} KB | {file_type}")
print("-" * 80)

print("\nData export completed successfully!")
print("   - All visualization data has been saved")
print("   - Compatible with Python, Excel, and other data analysis tools")
print("   - Ready for style adjustments and further analysis")
print("\nTips:")
print("   1. JSON files for programmatic processing with clear structure")
print("   2. CSV files for spreadsheet applications for easy viewing")
print("   3. Use pandas.read_csv() or json.load() for quick loading")
print("="*80)

## SHAP Decision Plot with All Features

Generate a single decision plot that includes all model features, with one-hot encoded categorical features consolidated into aggregated groups for cleaner visualization.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("SHAP DECISION PLOT - ALL FEATURES WITH CONSOLIDATED CATEGORICAL ENCODING")
print("="*80)

# Create a mapping to consolidate one-hot encoded categorical features
consolidated_feature_names = []
feature_indices_map = []  # Maps consolidated index to original indices

consolidated_idx = 0
original_idx = 0

while original_idx < len(feature_names):
    feat_name = feature_names[original_idx]
    
    # Check if this is a one-hot encoded categorical feature
    is_onehot = any(feat_name.startswith(cat_feat + '_') for cat_feat in cat_features)
    
    if is_onehot:
        # Find which categorical feature this belongs to
        cat_prefix = None
        for cat_feat in cat_features:
            if feat_name.startswith(cat_feat + '_'):
                cat_prefix = cat_feat
                break
        
        # Consolidate all features with the same prefix into one
        consolidated_name = f"{cat_prefix} (categorical)"
        category_indices = []
        
        start_idx = original_idx
        while original_idx < len(feature_names):
            if feature_names[original_idx].startswith(cat_prefix + '_'):
                category_indices.append(original_idx)
                original_idx += 1
            else:
                break
        
        consolidated_feature_names.append(consolidated_name)
        feature_indices_map.append(category_indices)
        consolidated_idx += 1
    else:
        # Keep non-categorical features as-is
        consolidated_feature_names.append(feat_name)
        feature_indices_map.append([original_idx])
        consolidated_idx += 1
        original_idx += 1

print(f"\nOriginal number of features: {len(feature_names)}")
print(f"Consolidated number of features: {len(consolidated_feature_names)}")
print(f"\nConsolidated feature groups:")
for i, feat in enumerate(consolidated_feature_names):
    num_original = len(feature_indices_map[i])
    if num_original > 1:
        print(f"  {feat} (aggregates {num_original} one-hot encoded features)")
    else:
        print(f"  {feat}")

# Compute consolidated SHAP values by averaging across one-hot encoded indices
consolidated_shap_values = np.zeros((shap_values.shape[0], len(consolidated_feature_names)))
consolidated_x_values = np.zeros((X_preprocessed.shape[0], len(consolidated_feature_names)))

for cons_idx, original_indices in enumerate(feature_indices_map):
    if len(original_indices) > 1:
        # Average SHAP values across one-hot encoded features
        consolidated_shap_values[:, cons_idx] = np.mean(
            np.abs(shap_values[:, original_indices]), axis=1
        )
        # For features, take mean of encoding values
        consolidated_x_values[:, cons_idx] = np.mean(
            X_preprocessed[:, original_indices], axis=1
        )
    else:
        # Single feature - use as-is
        consolidated_shap_values[:, cons_idx] = shap_values[:, original_indices[0]]
        consolidated_x_values[:, cons_idx] = X_preprocessed[:, original_indices[0]]

print(f"\n[Generating] Decision plot with {len(consolidated_feature_names)} consolidated features...")

# Create decision plot with all consolidated features
fig, ax = plt.subplots(figsize=(14, 8))

plt.sca(ax)
shap.decision_plot(
    expected_value,
    consolidated_shap_values,
    consolidated_x_values,
    feature_names=consolidated_feature_names,
    show=False
)

# Save figure in multiple formats
fig_name = "10_shap_decision_all_features"
fig.savefig(shap_viz_dir / f'{fig_name}.svg', format='svg', dpi=300, transparent=True, bbox_inches='tight')
fig.savefig(shap_viz_dir / f'{fig_name}.pdf', format='pdf', dpi=300, transparent=True, bbox_inches='tight')
fig.savefig(shap_viz_dir / f'{fig_name}.png', format='png', dpi=300, transparent=True, bbox_inches='tight')
#plt.close()

# Export decision plot data
decision_plot_data = {
    'title': 'SHAP Decision Plot - All Features with Consolidated Categorical Encoding',
    'base_value': float(expected_value),
    'num_consolidated_features': len(consolidated_feature_names),
    'consolidated_feature_names': consolidated_feature_names,
    'feature_consolidation_map': {
        consolidated_feature_names[i]: {
            'original_indices': feature_indices_map[i],
            'original_feature_names': [feature_names[j] for j in feature_indices_map[i]],
            'num_original_features': len(feature_indices_map[i])
        }
        for i in range(len(consolidated_feature_names))
    },
    'sample_consolidated_shap_values': consolidated_shap_values[:10].tolist(),
    'sample_feature_values': consolidated_x_values[:10].tolist(),
    'mean_absolute_shap_by_feature': np.abs(consolidated_shap_values).mean(axis=0).tolist(),
}

decision_file = data_export_dir / f'{fig_name}.json'
with open(decision_file, 'w') as f:
    json.dump(decision_plot_data, f, indent=2)

print(f"\n✓ Decision plot generated and saved")
print(f"  - Visualization: {shap_viz_dir / fig_name}.(svg/pdf/png)")
print(f"  - Data export: {decision_file}")



In [ ]:
ax.lines.count

In [ ]:
# ============================================================================
# SHAP Force Plot Generator with Encoder Integration
# ============================================================================
# This is a standalone cell that loads the model and generates predicted 
# force plots from raw (unencoded) feature inputs. No context dependencies.

import pandas as pd
import numpy as np
import shap
import joblib
import matplotlib.pyplot as plt
from pathlib import Path

print("=" * 80)
print("SHAP Force Plot Generator - Model Inference Interface")
print("=" * 80)

# Load pre-trained model, preprocessor, and SHAP explainer from files
model_output_dir = Path('ml_models')
shap_viz_dir = Path('shap_visualizations')

print("\n[Loading] Model components from files...")
best_model = joblib.load(model_output_dir / 'final_optimized_model.pkl')
preprocessor = joblib.load(model_output_dir / 'preprocessor.pkl') if (model_output_dir / 'preprocessor.pkl').exists() else None
explainer = joblib.load(model_output_dir / 'shap_explainer.pkl')
shap_values_data = np.load(model_output_dir / 'shap_values.npy', allow_pickle=True)

print("✓ Model loaded")
print("✓ SHAP explainer loaded")

# Get feature information
expected_value = explainer.expected_value if isinstance(explainer.expected_value, (int, float)) else explainer.expected_value[0]

# Feature definitions (must match training)
num_features = ['temperature_C', 'time_min', 'hmim_mM', 'zn_source_mM', 'ratio']
cat_features = ['counterion', 'solvent_name']
ord_features = ['stirring']
stirring_order = ['No', 'Initial', 'Yes', 'Yes dropwise']

# Recreate preprocessing pipeline if not saved
if preprocessor is None:
    from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
    from sklearn.compose import ColumnTransformer
    
    print("[Recreating] Preprocessing pipeline...")
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', 'passthrough', num_features),
            ('ord', OrdinalEncoder(categories=[stirring_order], handle_unknown='use_encoded_value', unknown_value=-1), ord_features),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
        ],
        remainder='passthrough'
    )
    # Fit on training data (need to load it)
    df_train = pd.read_csv('fit_data.csv').dropna(subset=['size_nm'])
    X_train = df_train[num_features + cat_features + ord_features]
    preprocessor.fit(X_train)

# Get feature names after preprocessing
feature_names = (num_features + ord_features + 
                list(preprocessor.named_transformers_['cat'].get_feature_names_out(cat_features)))

print(f"✓ Feature encoder initialized with {len(feature_names)} features")
print("=" * 80)

# ============================================================================
# Function: Generate Force Plot from Raw Inputs
# ============================================================================

def predict_and_plot(raw_inputs, sample_name="Custom Sample"):
    """
    Generate prediction and force plot from raw (unencoded) feature values.
    
    Parameters:
    -----------
    raw_inputs : dict
        Dictionary with raw feature values, e.g.:
        {
            'temperature_C': 50,
            'time_min': 180,
            'hmim_mM': 0.01,
            'zn_source_mM': 0.001,
            'ratio': 0.1077,
            'stirring': 'Yes',
            'counterion': 'NO3-',
            'solvent_name': 'Methanol'
        }
    sample_name : str
        Name for this sample (used in output files)
    
    Returns:
    --------
    dict : Prediction results with keys: 'prediction', 'shap_values', 'encoded_features'
    """
    
    print(f"\n{'=' * 80}")
    print(f"Generating prediction for: {sample_name}")
    print(f"{'=' * 80}\n")
    
    # Create DataFrame from raw inputs
    df_input = pd.DataFrame([raw_inputs])
    
    print("Raw input values:")
    for key, val in raw_inputs.items():
        print(f"  • {key}: {val}")
    
    # Encode features using preprocessor
    X_encoded = preprocessor.transform(df_input)
    
    # Generate prediction
    prediction = best_model.predict(X_encoded)[0]
    
    print(f"\n[Result] Predicted particle size: {prediction:.4f} nm")
    
    # Generate SHAP values for this sample
    sample_shap_values = explainer.shap_values(X_encoded)[0]
    
    # Create force plot
    print(f"\n[Generating] Force plot visualization...")
    
    # Generate interactive HTML force plot
    force_plot = shap.force_plot(
        expected_value,
        sample_shap_values,
        X_encoded[0],
        feature_names=feature_names,
        link='logit',
        matplotlib=False,
        show=False
    )
    
    # Save as interactive HTML
    html_content = shap.getjs() + force_plot.html()
    html_output_file = shap_viz_dir / f'force_plot_{sample_name.replace(" ", "_")}.html'
    with open(html_output_file, 'w') as f:
        f.write(html_content)
    print(f"✓ HTML saved: {html_output_file}")
    
    # Generate custom force plot visualization (editable SVG)
    base_name = f'force_plot_{sample_name.replace(" ", "_")}'
    
    # Create figure for custom force plot
    fig, ax = plt.subplots(figsize=(16, 6), dpi=100)
    
    # Sort contributions by absolute value
    contrib_indices = np.argsort(np.abs(sample_shap_values))[::-1][:12]  # Top 12 features
    top_shap_values = sample_shap_values[contrib_indices]
    top_features = [feature_names[i] for i in contrib_indices]
    
    # Calculate cumulative values for waterfall effect
    cumsum = np.concatenate([[expected_value], expected_value + np.cumsum(top_shap_values)])
    
    # Create waterfall chart
    colors = ['red' if x < 0 else 'green' for x in top_shap_values]
    
    # Plot base value
    ax.barh(0, expected_value, height=0.6, color='lightblue', edgecolor='navy', linewidth=2, label='Base value')
    
    # Plot feature contributions
    for i, (feat, val, color) in enumerate(zip(top_features, top_shap_values, colors)):
        y_pos = i + 1
        start_val = cumsum[i]
        ax.barh(y_pos, val, left=start_val, height=0.6, color=color, alpha=0.7, edgecolor='black', linewidth=1.5)
        
        # Add value labels
        mid_point = start_val + val / 2
        ax.text(mid_point, y_pos, f'{val:+.3f}', ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    
    # Plot final prediction
    final_y = len(top_features) + 1
    ax.barh(final_y, prediction, height=0.6, color='orange', edgecolor='darkred', linewidth=2, label='Prediction')
    ax.text(prediction/2, final_y, f'{prediction:.3f} nm', ha='center', va='center', fontsize=10, fontweight='bold', color='white')
    
    # Formatting
    ax.set_yticks(range(len(top_features) + 2))
    ax.set_yticklabels(['Base'] + top_features + ['Prediction'], fontsize=10)
    ax.set_xlabel('Impact on Output (nm)', fontsize=12, fontweight='bold')
    ax.set_title(f'SHAP Force Plot: {sample_name}\n(Feature Contributions to Particle Size Prediction)', 
                 fontsize=13, fontweight='bold', pad=15)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.legend(loc='lower right', fontsize=10)
    ax.invert_yaxis()
    
    # Tight layout
    plt.tight_layout()
    
    # Save as editable SVG (vector format)
    svg_file = shap_viz_dir / f'{base_name}.svg'
    fig.savefig(svg_file, format='svg', bbox_inches='tight', dpi=100)
    print(f"✓ SVG saved (editable): {svg_file}")
    
    # Also save as PDF
    pdf_file = shap_viz_dir / f'{base_name}.pdf'
    fig.savefig(pdf_file, format='pdf', bbox_inches='tight', dpi=100)
    print(f"✓ PDF saved: {pdf_file}")
    
    plt.close(fig)
    
    output_file = html_output_file
    
    # Print feature contributions
    print(f"\n[Feature Contributions] (Top 5)")
    contrib_df = pd.DataFrame({
        'Feature': feature_names,
        'SHAP Value': sample_shap_values,
        'Abs SHAP': np.abs(sample_shap_values)
    }).sort_values('Abs SHAP', ascending=False)
    
    for idx, row in contrib_df.head(5).iterrows():
        direction = "↑ UP" if row['SHAP Value'] > 0 else "↓ DOWN"
        print(f"  {row['Feature']:30s} {direction:8s} {row['SHAP Value']:+.4f}")
    
    result = {
        'prediction': prediction,
        'base_value': expected_value,
        'shap_values': sample_shap_values,
        'encoded_features': X_encoded[0],
        'output_file': {
            'html': html_output_file,
            'svg': svg_file,
            'pdf': pdf_file
        },
        'contributions_df': contrib_df
    }
    
    return result

# ============================================================================
# Example: User Input Section (Modify these values)
# ============================================================================

# Example 1: Predict for a custom sample
sample_1 = {
    'temperature_C': 50,
    'time_min': 180,
    'hmim_mM': 0.01,
    'zn_source_mM': 0.001,
    'ratio': 0.108,
    'stirring': 'Yes',
    'counterion': 'NO3-',
    'solvent_name': 'Methanol'
}

result_1 = predict_and_plot(sample_1, "sample_1")

# Example 2: Try another combination

target_ = pd.read_csv('GA_optimization_result_100nm.csv').iloc[0].to_dict()
sample_2 = {
    'temperature_C': target_.get('temperature_C',0),
    'time_min': target_.get('time_min',0),
    'hmim_mM': target_.get('hmim_mM',0),
    'zn_source_mM': target_.get('zn_source_mM',0),
    'ratio': target_.get('ratio',0),
    'stirring': target_.get('stirring',0),
    'counterion': target_.get('counterion',0),
    'solvent_name': target_.get('solvent_name',0)
}

result_2 = predict_and_plot(sample_2, "sample_2")

print("\n" + "=" * 80)
print("✨ Predictions completed!")
print("=" * 80)
print(f"\nGenerated HTML files in: {shap_viz_dir}/")
print("Open these files in your browser to view interactive force plots.")


In [ ]:
target_ = pd.read_csv('GA_optimization_result_100nm.csv').iloc[0].to_dict()
sample_2 = {
    'temperature_C': target_.get('temperature_C',0),
    'time_min': target_.get('time_min',0),
    'hmim_mM': target_.get('hmim_mM',0),
    'zn_source_mM': target_.get('zn_source_mM',0),
    'ratio': target_.get('ratio',0),
    'stirring': target_.get('stirring',0),
    'counterion': target_.get('counterion',0),
    'solvent_name': target_.get('solvent_name',0)
}
sample_2